# DQN

In [40]:

import torch
from open_spiel.python import rl_environment
from open_spiel.python.pytorch import dqn

# Environment
env = rl_environment.Environment("pentago")
num_actions = env.action_spec()["num_actions"]
state_size = env.observation_spec()["info_state"][0]

### Adversarial Training

In [39]:


agents = [
    dqn.DQN(
        player_id=i,
        state_representation_size=state_size,
        num_actions=num_actions,
        hidden_layers_sizes=[256, 128],
        replay_buffer_capacity=100_000,
        batch_size=128,
        learning_rate=1e-3,
        update_target_network_every=500,
        learn_every=50,
        discount_factor=0.99,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_duration=50_000,
        optimizer_str="adam"
    )
    for i in range(2)
]

# Training loop
num_episodes = 80_000

for ep in range(num_episodes):
    time_step = env.reset()

    while not time_step.last():
        current_player = time_step.observations["current_player"]
        agent_output = agents[current_player].step(time_step)
        time_step = env.step([agent_output.action])

    # Let both agents observe the terminal state
    for agent in agents:
        agent.step(time_step)

    # Logging
    if ep % 500 == 0:
        print(f"Episode {ep}/{num_episodes} | "
              f"P0 loss: {agents[0].loss} | "
              f"P1 loss: {agents[1].loss} | ")

# Save the trained agents
torch.save(agents[0]._q_network.state_dict(), "agent0.pt")
torch.save(agents[1]._q_network.state_dict(), "agent1.pt")
print("Training complete!")

Episode 0/80000 | P0 loss: None | P1 loss: None | 


/Users/colegiusto/College/ECE270/FinalProject/.venv/lib/python3.12/site-packages/open_spiel/python/pytorch/dqn.py:338: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:353.)
  predictions = self._q_values[list(action_indices)]


Episode 500/80000 | P0 loss: 1.3041613101959229 | P1 loss: 0.6947082877159119 | 
Episode 1000/80000 | P0 loss: 19.868276596069336 | P1 loss: 15.475947380065918 | 
Episode 1500/80000 | P0 loss: 1.4924412965774536 | P1 loss: 0.39862725138664246 | 


KeyboardInterrupt: 

### Random Training

In [51]:
from open_spiel.python.algorithms import random_agent


agents = [
    dqn.DQN(
        player_id=0,
        state_representation_size=state_size,
        num_actions=num_actions,
        hidden_layers_sizes=(256,128),
        replay_buffer_capacity=100_000,
        batch_size=128,
        learning_rate=1e-3,
        update_target_network_every=500,
        learn_every=50,
        discount_factor=0.99,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay_duration=50_000,
        optimizer_str="adam",
        loss_str="huber",
    ),
    random_agent.RandomAgent(
        player_id=1,
        num_actions=num_actions
    )
]

# Training loop
num_episodes = 80_000

for ep in range(num_episodes):
    time_step = env.reset()

    while not time_step.last():
        current_player = time_step.observations["current_player"]
        agent_output = agents[current_player].step(time_step)
        time_step = env.step([agent_output.action])

    # Let both agents observe the terminal state
    for agent in agents:
        agent.step(time_step)

    # Logging
    if ep % 500 == 0:
        print(f"Episode {ep}/{num_episodes} | "
              f"P0 loss: {agents[0].loss} | ")

# Save the trained agents
torch.save(agents[0]._q_network.state_dict(), "agent0_vs_rand.pt")

print("Training complete!")

Episode 0/80000 | P0 loss: None | 
Episode 500/80000 | P0 loss: 0.44489261507987976 | 
Episode 1000/80000 | P0 loss: 1.9759788513183594 | 
Episode 1500/80000 | P0 loss: 12.318099975585938 | 
Episode 2000/80000 | P0 loss: 12.656089782714844 | 
Episode 2500/80000 | P0 loss: 1.9750455617904663 | 
Episode 3000/80000 | P0 loss: 0.030852552503347397 | 
Episode 3500/80000 | P0 loss: 0.017693376168608665 | 
Episode 4000/80000 | P0 loss: 0.02515614777803421 | 
Episode 4500/80000 | P0 loss: 0.04680854082107544 | 
Episode 5000/80000 | P0 loss: 0.042162004858255386 | 
Episode 5500/80000 | P0 loss: 0.027147259563207626 | 
Episode 6000/80000 | P0 loss: 0.04060938209295273 | 
Episode 6500/80000 | P0 loss: 0.049145374447107315 | 
Episode 7000/80000 | P0 loss: 0.02817058563232422 | 
Episode 7500/80000 | P0 loss: 0.024051839485764503 | 
Episode 8000/80000 | P0 loss: 0.049185317009687424 | 
Episode 8500/80000 | P0 loss: 0.04023437947034836 | 
Episode 9000/80000 | P0 loss: 0.028640300035476685 | 
Episode 

## AlphaZero

RecursionError: maximum recursion depth exceeded while calling a Python object

## Evaluate

### Vs Random

In [ ]:
from open_spiel.python.algorithms import random_agent

def eval_vs_random(trained_agent, num_episodes=1000, agent_player_id=0):
    rand_agent = random_agent.RandomAgent(
        player_id=1 - agent_player_id,
        num_actions=num_actions
    )
    
    wins, losses, draws = 0, 0, 0

    for ep in range(num_episodes):
        time_step = env.reset()

        while not time_step.last():
            current_player = time_step.observations["current_player"]
            if current_player == agent_player_id:
                action = trained_agent.step(time_step, is_evaluation=True).action
            else:
                action = rand_agent.step(time_step).action
            time_step = env.step([action])

        # Check result
        final_returns = env._state.returns()
        if final_returns[agent_player_id] > 0:
            wins += 1
        elif final_returns[agent_player_id] < 0:
            losses += 1
        else:
            draws += 1

        if (ep + 1) % 100 == 0:
            print(f"Episode {ep+1}: W={wins} L={losses} D={draws} | "
                  f"Winrate: {wins/(ep+1):.2%}")

    print(f"\nFinal over {num_episodes} games:")
    print(f"  Wins:   {wins}  ({wins/num_episodes:.2%})")
    print(f"  Losses: {losses}  ({losses/num_episodes:.2%})")
    print(f"  Draws:  {draws}  ({draws/num_episodes:.2%})")

# Evaluate after training
eval_vs_random(agents[0], num_episodes=1000, agent_player_id=0)

Episode 100: W=96 L=4 D=0 | Winrate: 96.00%
Episode 200: W=194 L=6 D=0 | Winrate: 97.00%
Episode 300: W=293 L=6 D=1 | Winrate: 97.67%
Episode 400: W=387 L=11 D=2 | Winrate: 96.75%
Episode 500: W=481 L=15 D=4 | Winrate: 96.20%
Episode 600: W=577 L=18 D=5 | Winrate: 96.17%
Episode 700: W=676 L=18 D=6 | Winrate: 96.57%
Episode 800: W=771 L=22 D=7 | Winrate: 96.38%
Episode 900: W=870 L=23 D=7 | Winrate: 96.67%
Episode 1000: W=967 L=25 D=8 | Winrate: 96.70%

Final over 1000 games:
  Wins:   967  (96.70%)
  Losses: 25  (2.50%)
  Draws:  8  (0.80%)


### Vs Optimal

In [92]:
import requests
import numpy as np

def get_perfect_pentago_id(env):
    state = env.get_state
    obs = np.array(state.observation_tensor()).reshape(3, 6, 6)
    
    
    # Quadrants: Lower Left (LL), Upper Left (UL), Lower Right (LR), Upper Right (UR)
    # This order matches the bits 0-15, 16-31, 32-47, 48-63
    quad_defs = [
    (range(0, 3), range(0, 3)),  # UL (i=0, bits 0–15)
    (range(0, 3), range(3, 6)),  # UR (i=1, bits 16–31)
    (range(3, 6), range(0, 3)),  # LL (i=2, bits 32–47)
    (range(3, 6), range(3, 6)),  # LR (i=3, bits 48–63)
]
    
    final_id = 0
    for i, (r_range, c_range) in enumerate(quad_defs):
        quad_sum = 0
        # x-major loop
        power = 0
        for r in r_range:
            for c in c_range:
                # 1 for Black (Player 0), 2 for White (Player 1)
                cell_val = 0
                if obs[0][r][c] == 1: cell_val = 1
                elif obs[1][r][c] == 1: cell_val = 2
                
                quad_sum += cell_val * (3**power)
                power += 1
        
        final_id |= (quad_sum << (16 * i))
    
    return final_id

def get_perfect_move(env):
    state_id = get_perfect_pentago_id(env)
    print(state_id)
    URL = f"https://us-central1-naml-148801.cloudfunctions.net/pentago/{state_id}"
    res = requests.get(URL)
    return res

def eval_vs_optimal(trained_agent: dqn.DQN, num_episodes=1000, agent_player_id=0):
    wins, losses, draws = 0, 0, 0

    for ep in range(num_episodes):
        time_step = env.reset()

        while not time_step.last():
            current_player = time_step.observations["current_player"]
            if current_player == agent_player_id:
                action = trained_agent.step(time_step, is_evaluation=True).action
            else:
                print(env.get_state)
                print(get_perfect_move(env))
                return
            time_step = env.step([action])


eval_vs_optimal(agents[0], num_episodes=1000, agent_player_id=0)

    > t     u <
    a b c d e f
v 1 . . . . . . v
s 2 . . . . . . v
  3 . . . . . .  
  4 . . . . . .  
z 5 . O . . . . w
^ 6 . . . . . . ^
    > y     x <

695784701952
<Response [404]>
